# HDD 11 — One offline replay fine-tuning trial, acceptance checks and rollback

This performs **one experimental offline encoder update**, not a drift-triggered production adaptation. Notebook 10 supplies screened existing-partition candidates; no new temporal regime has been validated.

The original checkpoint, preprocessing, partition model, prototypes and thresholds remain immutable. A separate candidate is trained for exactly five epochs with equal counts of new and historical replay windows. No early stopping or hyperparameter choice uses September. September acceptance scores are opened here for one predeclared candidate comparison. June and failure labels remain closed.

A candidate can be rejected. Passing these pilot checks does not establish real failure sensitivity, lower false-positive rates, operational safety, drift recovery or multi-model generalisation. Regardless of the decision, the in-memory model is restored to the frozen checkpoint and verified.

In [1]:
from pathlib import Path
import os,json,hashlib,platform
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import torch,joblib
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from hdbscan.prediction import approximate_predict
from tqdm.auto import tqdm
from IPython.display import display
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
ABLATION_ID='representation_ablation_v1_seed42'
SCORE_ID='partition_scoring_v1'
EXPANSION_ID='q3_development_expansion_v1'
AUDIT_ID='offline_replay_adaptation_v1'
OUT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
OUT.mkdir(parents=True,exist_ok=True)
INPUTS={}
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def remember(p):
    assert p.is_file(),f'Missing file: {p}'
    INPUTS[str(p.relative_to(COMPONENT_ROOT))]=sha256(p)
    return p
def load_json(p):return json.loads(remember(p).read_text())
exp_out=COMPONENT_ROOT/'results'/RUN_ID/EXPANSION_ID
exp=load_json(exp_out/'expansion_complete.json')
assert exp['analysis_complete'] and exp['dataset_feasible']
assert not exp['failure_labels_read'] and not exp['sealed_June_test_opened']
assert not exp['acceptance_scores_inspected']
splits=load_json(exp_out/'q3_serials.json')
assert sha256(exp_out/'q3_serials.json')==exp['split_sha256']
DATA=COMPONENT_ROOT/'data/features'/RUN_ID/EXPANSION_ID/'adaptation'
# Verify only adaptation artifacts. Acceptance files are not opened here.
for rel,digest in exp['artifact_sha256'].items():
    if Path(rel).parts[-2:-1]==('adaptation',):
        assert sha256(remember(COMPONENT_ROOT/rel))==digest
sm=COMPONENT_ROOT/'models/trained'/RUN_ID/SCORE_ID
score_complete=load_json(COMPONENT_ROOT/'results'/RUN_ID/SCORE_ID/'scoring_complete.json')
spec_path=sm/'scoring_specification.json';spec=load_json(spec_path)
assert sha256(spec_path)==score_complete['specification_sha256']
assert score_complete['analysis_complete'] and not score_complete['sealed_test_opened'] and not score_complete['real_failure_labels_used']
assert spec['transition_weight']==0
ABLATION_MODELS=COMPONENT_ROOT/'models/trained'/RUN_ID/ABLATION_ID
ENC_SETTINGS=load_json(ABLATION_MODELS/'settings.json')
delta_path=ABLATION_MODELS/'change_preprocessing.json';delta=load_json(delta_path)
prep_path=COMPONENT_ROOT/'models/trained'/f'{RUN_ID}_daily_preprocessing.json';prep=load_json(prep_path)
ablation_path=COMPONENT_ROOT/'results'/RUN_ID/ABLATION_ID/'representation_ablation_complete.json'
ablation=load_json(ablation_path)
assert sha256(ablation_path)==spec['ablation_complete_sha256']
assert sha256(prep_path)==ablation['settings']['daily_preprocessing_sha256']
for path in [ABLATION_MODELS/'settings.json',delta_path]:
    assert sha256(path)==ablation['artifact_sha256'][str(path.relative_to(COMPONENT_ROOT))]
FEATURES=prep['features'];T=30;F=len(FEATURES)
assert FEATURES==delta['features']==exp['settings']['features'] and F==5
TEMPERATURE=FEATURES.index('smart_194_raw')
SECTORS=[FEATURES.index(f) for f in ['smart_5_raw','smart_197_raw','smart_198_raw']]
pipeline_path=COMPONENT_ROOT/spec['upstream_partition_pipeline']
assert sha256(remember(pipeline_path))==spec['partition_pipeline_sha256']
pipeline=joblib.load(pipeline_path) # Load only your own trusted model artifact.
assert sha256(remember(sm/'prototypes.npz'))==spec['prototype_sha256']
with np.load(sm/'prototypes.npz') as a:
    centers=a['centers'];radii=a['radii'];prototype_labels=a['labels']
assert np.isfinite(radii).all() and (radii>0).all()
THRESHOLDS=spec['thresholds'];NORMALISERS=spec['normalisation']
SETTINGS={'minimum_strength':.5,'screen_span_calendar_days':14,'minimum_daily_drives':30,
    'minimum_block_windows':200,'minimum_partition_share':.8,
    'sector_guard':'exclude any positive sector count or positive daily sector change anywhere in window',
    'score_guard':'exclude any global R/E/D/RD alert','upstream_expansion_sha256':sha256(exp_out/'expansion_complete.json'),
    'score_specification_sha256':sha256(spec_path),'automatic_adaptation':False}
p=OUT/'frozen_input_settings.json'
if p.exists():assert json.loads(p.read_text())==SETTINGS,'Changed settings: use a new AUDIT_ID.'
p.write_text(json.dumps(SETTINGS,indent=2))
print('Frozen artifacts verified; acceptance scores remain unopened.')
AUDIT_OUT=COMPONENT_ROOT/'results'/RUN_ID/'q3_frozen_candidate_audit_v1'
audit_record=load_json(AUDIT_OUT/'q3_frozen_audit_complete.json')
assert audit_record['analysis_complete'] and audit_record['offline_candidate_data_available']
assert not audit_record['acceptance_scores_inspected'] and not audit_record['sealed_test_opened']
for name,digest in audit_record['artifact_sha256'].items():
    assert sha256(remember(AUDIT_OUT/name))==digest, f'Changed Notebook 10 artifact: {name}'
POLICY=load_json(AUDIT_OUT/'offline_adaptation_policy.json')
assert POLICY['candidate_comparisons_allowed']==1 and POLICY['fine_tuning_epochs']==5
assert POLICY['historical_replay_fraction']==.5
assert not POLICY['thresholds_refitted'] and not POLICY['partition_model_refitted']
assert not POLICY['acceptance_scores_inspected'] and not POLICY['automatic_promotion_enabled']
TRAINING={'seed':POLICY['seed'],'epochs':POLICY['fine_tuning_epochs'],'learning_rate':POLICY['learning_rate'],
    'new_windows_per_batch':32,'weight_decay':.0001,'point_mask_probability':.15,'span_days':3,
    'gradient_clip_norm':1.,'maximum_partition_assignment_change_fraction':.05,
    'old_retention_minimum_partition_windows':30,'old_retention_minimum_partition_drives':5,
    'upstream_policy_sha256':sha256(AUDIT_OUT/'offline_adaptation_policy.json'),
    'upstream_audit_complete_sha256':sha256(AUDIT_OUT/'q3_frozen_audit_complete.json')}
training_path=OUT/'training_settings.json'
if training_path.exists():assert json.loads(training_path.read_text())==TRAINING,'Changed trial: do not reuse this ID or the consumed acceptance set for tuning.'
training_path.write_text(json.dumps(TRAINING,indent=2))
MODEL_OUT=COMPONENT_ROOT/'models/trained'/RUN_ID/AUDIT_ID
MODEL_OUT.mkdir(parents=True,exist_ok=True)
print('One fixed offline trial. September will be evaluated after the candidate checkpoint is finalized.')


c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Frozen artifacts verified; acceptance scores remain unopened.
One fixed offline trial. September will be evaluated after the candidate checkpoint is finalized.


## 1. Restore and reproduce the frozen encoder

This is the same model definition and deterministic five-pass inference used in Notebook 10. Only the separate candidate model will be trained.

In [2]:
class LevelChangeEncoder(nn.Module):
    def __init__(self):
        super().__init__();width=ENC_SETTINGS['width']
        self.input_projection=nn.Linear(5*F,width)
        self.position=nn.Parameter(torch.empty(1,T,width));nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(width,ENC_SETTINGS['heads'],ENC_SETTINGS['feedforward'],ENC_SETTINGS['dropout'],
            activation='gelu',batch_first=True,norm_first=False)
        self.encoder=nn.TransformerEncoder(block,ENC_SETTINGS['layers'],enable_nested_tensor=False)
        for layer in self.encoder.layers:
            for parameter in layer.parameters():
                if parameter.dim()>1:nn.init.xavier_uniform_(parameter)
        self.decoder=nn.Linear(width,F)
    def forward(self,levels,changes,hide):
        visible=~hide
        delta_available=torch.zeros_like(visible)
        delta_available[:,1:]=visible[:,1:] & visible[:,:-1]
        channels=torch.cat([levels.masked_fill(hide,0.),changes.masked_fill(~delta_available,0.),
            torch.ones_like(levels),hide.float(),delta_available.float()],dim=-1)
        hidden=self.encoder(self.input_projection(channels)+self.position[:,:levels.shape[1]])
        return self.decoder(hidden),hidden.mean(1)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cpu':torch.set_num_threads(max(1,min(4,os.cpu_count() or 1)))
model=LevelChangeEncoder().to(DEVICE)
checkpoint_path=ABLATION_MODELS/'level_change_encoder.pt'
assert checkpoint_path==COMPONENT_ROOT/spec['upstream_encoder_checkpoint']
assert sha256(remember(checkpoint_path))==spec['checkpoint_sha256']
checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
expected_input_hash=hashlib.sha256((sha256(ABLATION_MODELS/'settings.json')+sha256(delta_path)).encode()).hexdigest()
assert checkpoint['model_input_hash']==expected_input_hash
model.load_state_dict(checkpoint['state_dict']);model.eval()
COUNT_INDICES=[FEATURES.index(f) for f in prep['log_features']]
grid=(torch.arange(T)[:,None]+torch.arange(F)[None,:])%F
PARTITIONS=[grid.eq(k) for k in range(F)]
assert torch.stack(PARTITIONS).sum(0).eq(1).all()

def raw_channels(raw):
    levels=raw.astype(np.float64,copy=True)
    for j in COUNT_INDICES:levels[...,j]=np.log1p(levels[...,j])
    levels=((levels-np.array(prep['center']))/np.array(prep['scale'])).astype(np.float32)
    change=np.diff(raw.astype(np.float64),axis=1)
    for j in COUNT_INDICES:change[...,j]=np.sign(change[...,j])*np.log1p(np.abs(change[...,j]))
    changes=np.zeros_like(raw,dtype=np.float32)
    changes[:,1:]=((change-np.array(delta['center']))/np.array(delta['scale'])).astype(np.float32)
    return levels,changes

@torch.no_grad()
def infer_raw(raw,progress=False):
    assert np.isfinite(raw).all() and (raw>=0).all()
    levels,changes=raw_channels(raw);embeddings=[];feature_errors=[];last_day=[]
    model.eval()
    loader=DataLoader(TensorDataset(torch.from_numpy(levels),torch.from_numpy(changes)),batch_size=ENC_SETTINGS['batch_size'],shuffle=False,num_workers=0)
    for x,d in (tqdm(loader,desc='Frozen encoder inference',leave=False) if progress else loader):
        x=x.to(DEVICE);d=d.to(DEVICE);errors=torch.zeros_like(x);latent=None
        for mask in PARTITIONS:
            hide=mask.to(DEVICE).unsqueeze(0).expand_as(x)
            predicted,z=model(x,d,hide)
            errors+=torch.where(hide,(predicted-x).square(),torch.zeros_like(x));latent=z if latent is None else latent+z
        embeddings.append((latent/F).cpu().numpy());feature_errors.append(errors.mean(1).cpu().numpy());last_day.append(errors[:,-1].cpu().numpy())
    result={'embeddings':np.concatenate(embeddings),'feature_errors':np.concatenate(feature_errors),'last_day_errors':np.concatenate(last_day)}
    for value in result.values():assert np.isfinite(value).all()
    return result


original_root=COMPONENT_ROOT/'data/features'/RUN_ID
with np.load(remember(original_root/'development/windows_raw.npz')) as a:
    reference_raw=a['X'][:8].astype(np.float32)
with np.load(remember(original_root/ABLATION_ID/'new_encoder/development/representations.npz')) as a:
    expected={k:a[k][:8] for k in ['embeddings','feature_errors','last_day_errors']}
check=infer_raw(reference_raw)
for key in check:np.testing.assert_allclose(check[key],expected[key],rtol=1e-3,atol=1e-4)
print('Frozen encoder reproduces original saved outputs. No training performed.')

FROZEN_STATE={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
FROZEN_HASH=sha256(checkpoint_path)


Frozen encoder reproduces original saved outputs. No training performed.


In [3]:
def frozen_scores(result):
    embedding=np.ascontiguousarray(result['embeddings'],dtype=np.float64)
    z=np.ascontiguousarray(pipeline['pca'].transform(pipeline['scaler'].transform(embedding)),dtype=np.float64)
    labels,strength=approximate_predict(pipeline['clusterer'],z)
    distances=np.linalg.norm(z[:,None,:]-centers[None,:,:],axis=2)/radii[None,:]
    scores={'R':result['feature_errors'].mean(1),'E':result['last_day_errors'][:,TEMPERATURE],
        'D':distances.min(1)}
    norms={k:(np.log1p(scores[k])-NORMALISERS[k]['center'])/NORMALISERS[k]['scale'] for k in ['R','D']}
    scores['RD']=spec['settings']['reconstruction_weight']*norms['R']+spec['settings']['distance_weight']*norms['D']
    context=np.where((labels>=0)&(strength>=SETTINGS['minimum_strength'])&np.isin(labels,prototype_labels),labels,-1)
    assert all(np.isfinite(v).all() for v in scores.values())
    return scores,labels.astype(int),strength,context

def threshold_flags(values,context,name,policy):
    thresholds=np.full(len(values),THRESHOLDS[name]['global'],dtype=float)
    source=np.full(len(values),'global' if policy=='global' else 'global_fallback',dtype=object)
    if policy=='conditional':
        for label,value in THRESHOLDS[name]['groups'].items():
            selected=context==int(label);thresholds[selected]=value;source[selected]='partition_'+str(label)
    else:assert policy=='global'
    return values>thresholds,thresholds,source


# Historical replay indices were frozen in Notebook 08.
anchor_path=COMPONENT_ROOT/'results'/RUN_ID/'quarantine_replay_readiness_v1/historical_replay_anchor_index.csv'
anchors=pd.read_csv(remember(anchor_path),dtype={'serial_number':str},parse_dates=['end_date'])
train_meta=pd.read_parquet(remember(original_root/'train/metadata.parquet')).reset_index(drop=True)
with np.load(remember(original_root/'train/windows_raw.npz')) as a:train_raw=a['X'].astype(np.float32)
old_indices=anchors.source_row.to_numpy(dtype=int)
assert len(np.unique(old_indices))==len(old_indices)
assert (old_indices>=0).all() and (old_indices<len(train_raw)).all()
assert anchors.serial_number.tolist()==train_meta.iloc[old_indices].serial_number.tolist()
np.testing.assert_array_equal(anchors.end_date.to_numpy(dtype='datetime64[D]'),pd.to_datetime(train_meta.iloc[old_indices].end_date).to_numpy(dtype='datetime64[D]'))
old_raw=train_raw[old_indices]
new_meta=pd.read_parquet(remember(DATA/'metadata.parquet')).reset_index(drop=True)
with np.load(remember(DATA/'windows_raw.npz')) as a:
    new_raw=a['X'].astype(np.float32);assert a['features'].tolist()==FEATURES
candidate_index=pd.read_csv(remember(AUDIT_OUT/'offline_candidate_index.csv'),dtype={'serial_number':str},parse_dates=['start_date','end_date'])
new_indices=candidate_index.source_row.to_numpy(dtype=int)
assert len(new_indices)>=30 and candidate_index.serial_number.nunique()>=30
assert len(np.unique(new_indices))==len(new_indices)
assert (new_indices>=0).all() and (new_indices<len(new_raw)).all()
assert candidate_index.serial_number.tolist()==new_meta.iloc[new_indices].serial_number.tolist()
np.testing.assert_array_equal(candidate_index.end_date.to_numpy(dtype='datetime64[D]'),pd.to_datetime(new_meta.iloc[new_indices].end_date).to_numpy(dtype='datetime64[D]'))
assert set(candidate_index.serial_number).isdisjoint(anchors.serial_number)
assert set(candidate_index.serial_number).isdisjoint(splits['acceptance'])
new_fit=new_raw[new_indices]
assert not (new_fit[:,:,SECTORS]>0).any(), 'Sector-positive window entered the candidate set.'
for values in [new_fit,old_raw]:assert values.shape[1:]==(T,F) and np.isfinite(values).all() and (values>=0).all()
old_levels,old_changes=raw_channels(old_raw);new_levels,new_changes=raw_channels(new_fit)
print('New windows:',len(new_fit),'| new drives:',candidate_index.serial_number.nunique())
print('Historical replay pool:',len(old_raw),'| drives:',anchors.serial_number.nunique())
print('Historical retention is a training replay check, not an independent holdout.')


New windows: 188 | new drives: 97
Historical replay pool: 4885 | drives: 700
Historical retention is a training replay check, not an independent holdout.


## 2. Fit exactly one candidate with 50% historical replay

Each new-window batch is paired with an equally sized random historical batch, sampled with replacement from the frozen anchor pool. Train on masked observed levels, with change channels hidden where their endpoints are hidden. Save only the final fifth-epoch checkpoint; there is no acceptance-guided checkpoint selection.

An interrupted training run restarts this same trial from the immutable frozen checkpoint. A completed candidate is reused after checking its input identity.

In [4]:
import random
random.seed(TRAINING['seed']);np.random.seed(TRAINING['seed']);torch.manual_seed(TRAINING['seed'])
if torch.cuda.is_available():torch.cuda.manual_seed_all(TRAINING['seed'])
torch.use_deterministic_algorithms(True,warn_only=True)
trial_identity=hashlib.sha256((json.dumps(INPUTS,sort_keys=True)+json.dumps(TRAINING,sort_keys=True)+FROZEN_HASH).encode()).hexdigest()
candidate_path=MODEL_OUT/'candidate_encoder.pt'
def random_hide(shape,generator):
    b,t,f=shape
    hide=torch.rand(shape,generator=generator)<TRAINING['point_mask_probability']
    span=TRAINING['span_days']
    starts=torch.randint(0,t-span+1,(b,),generator=generator)
    channels=torch.randint(0,f,(b,),generator=generator)
    for i in range(b):hide[i,starts[i]:starts[i]+span,channels[i]]=True
    return hide
if candidate_path.exists():
    payload=torch.load(candidate_path,map_location='cpu',weights_only=True)
    assert payload['trial_identity']==trial_identity and payload['completed_epochs']==TRAINING['epochs'],'Different or incomplete candidate; do not compare another candidate on this acceptance reservation.'
    history=payload['history'];print('Reused completed candidate; no new trial trained.')
else:
    model.load_state_dict(FROZEN_STATE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=TRAINING['learning_rate'],weight_decay=TRAINING['weight_decay'])
    rng=np.random.default_rng(TRAINING['seed']);generator=torch.Generator().manual_seed(TRAINING['seed']+10)
    history=[];draws=[]
    for epoch in range(1,TRAINING['epochs']+1):
        model.train();total=0.;count=0;new_count=0;old_count=0
        permutation=rng.permutation(len(new_fit))
        for start in range(0,len(permutation),TRAINING['new_windows_per_batch']):
            ni=permutation[start:start+TRAINING['new_windows_per_batch']]
            oi=rng.integers(0,len(old_raw),size=len(ni))
            for a,b in zip(ni,oi):draws.append({'epoch':epoch,'new_source_row':int(new_indices[a]),'old_source_row':int(old_indices[b])})
            x=torch.from_numpy(np.concatenate([new_levels[ni],old_levels[oi]])).to(DEVICE)
            d=torch.from_numpy(np.concatenate([new_changes[ni],old_changes[oi]])).to(DEVICE)
            hide=random_hide(x.shape,generator).to(DEVICE)
            predicted,_=model(x,d,hide);errors=(predicted-x).square();loss=errors[hide].mean()
            assert torch.isfinite(loss),'Nonfinite candidate loss.'
            optimizer.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),TRAINING['gradient_clip_norm']);optimizer.step()
            total+=float(errors[hide].detach().sum());count+=int(hide.sum());new_count+=len(ni);old_count+=len(oi)
        assert new_count==old_count==len(new_fit)
        history.append({'epoch':epoch,'masked_training_mse':total/count,'new_draws':new_count,'historical_draws':old_count})
        print('Epoch',epoch,'| masked training MSE:',total/count)
    payload={'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
        'trial_identity':trial_identity,'completed_epochs':TRAINING['epochs'],'history':history}
    temporary=candidate_path.with_suffix('.pt.tmp');torch.save(payload,temporary);temporary.replace(candidate_path)
    pd.DataFrame(draws).to_csv(OUT/'training_replay_draws.csv',index=False)
CANDIDATE_STATE=payload['state_dict'];CANDIDATE_HASH=sha256(candidate_path)
STATES={'frozen':FROZEN_STATE,'candidate':CANDIDATE_STATE}
pd.DataFrame(history).to_csv(OUT/'candidate_training_history.csv',index=False)
assert sha256(checkpoint_path)==FROZEN_HASH
print('Final candidate saved separately; frozen checkpoint unchanged.')


Epoch 1 | masked training MSE: 0.0050480600671892065
Epoch 2 | masked training MSE: 0.004856334160974085
Epoch 3 | masked training MSE: 0.00539988877416468
Epoch 4 | masked training MSE: 0.005117480257534357
Epoch 5 | masked training MSE: 0.004668560147818212
Final candidate saved separately; frozen checkpoint unchanged.


## 3. Open September for the single declared comparison

The reservation ledger records this candidate and policy before acceptance inference. Rerunning the same fixed comparison is reproducible; substituting a different candidate after opening September is disallowed. The ledger cannot erase prior exposure: do not delete it to tune another model.

The acceptance group is not labelled clean. We measure reconstruction and alert burden, not false-positive rates. Coverage remains the 2,989 accepted windows from Notebook 09; rejected windows are not imputed or silently reintroduced.

In [5]:
accept_folder=COMPONENT_ROOT/'data/features'/RUN_ID/EXPANSION_ID/'acceptance'
# All acceptance bytes must match the reservation prepared before candidate fitting.
for path in [accept_folder/'metadata.parquet',accept_folder/'windows_raw.npz']:
    rel=str(path.relative_to(COMPONENT_ROOT))
    assert sha256(remember(path))==exp['artifact_sha256'][rel]
ledger_identity={'candidate_sha256':CANDIDATE_HASH,'trial_identity':trial_identity,
    'policy_sha256':sha256(AUDIT_OUT/'offline_adaptation_policy.json'),
    'training_settings_sha256':sha256(training_path),
    'acceptance_raw_sha256':sha256(accept_folder/'windows_raw.npz')}
ledger_path=exp_out/'acceptance_comparison_ledger.json'
if ledger_path.exists():assert json.loads(ledger_path.read_text())==ledger_identity,'Acceptance reservation already consumed by a different comparison. Keep that result; do not retune against this holdout.'
else:ledger_path.write_text(json.dumps(ledger_identity,indent=2))
accept_meta=pd.read_parquet(accept_folder/'metadata.parquet').reset_index(drop=True)
with np.load(accept_folder/'windows_raw.npz') as a:
    accept_raw=a['X'].astype(np.float32);assert a['features'].tolist()==FEATURES and a['observed_mask'].all()
assert len(accept_meta)==len(accept_raw)>0
assert set(accept_meta.serial_number).issubset(splits['acceptance'])
assert set(accept_meta.serial_number).isdisjoint(set(candidate_index.serial_number)|set(anchors.serial_number))
assert pd.to_datetime(accept_meta.end_date).min()>pd.to_datetime(candidate_index.end_date).max()

def cached_inference(name,tag,values):
    # Include value bytes and model hash to prevent reuse for the wrong perturbation/model.
    value_hash=hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()
    identity=hashlib.sha256((name+value_hash+(FROZEN_HASH if name=='frozen' else CANDIDATE_HASH)+sha256(prep_path)+sha256(delta_path)).encode()).hexdigest()
    path=OUT/f'{tag}_{name}_representations.npz';mp=path.with_suffix('.json')
    model.load_state_dict(STATES[name]);model.eval()
    if path.exists() and mp.exists():
        saved=json.loads(mp.read_text());assert saved['identity']==identity and saved['sha256']==sha256(path),'Changed inference inputs: do not overwrite trial caches.'
        with np.load(path) as a:return {k:a[k] for k in a.files}
    result=infer_raw(values,progress=True)
    np.savez_compressed(path,**result);mp.write_text(json.dumps({'identity':identity,'sha256':sha256(path)},indent=2))
    return result
REP={};SCORED={};METADATA={'old_replay':anchors[['serial_number','end_date']].copy(),'acceptance':accept_meta}
metrics=[]
for split,values in [('old_replay',old_raw),('acceptance',accept_raw)]:
    for name in STATES:
        result=cached_inference(name,split,values);REP[(split,name)]=result
        scores,labels,strength,context=frozen_scores(result);SCORED[(split,name)]=(scores,labels,strength,context)
        for detector,score in scores.items():
            for policy in ['global','conditional']:
                flags,thresholds,sources=threshold_flags(score,context,detector,policy)
                table=METADATA[split].copy();table['score']=score;table['threshold']=thresholds
                table['partition_label']=labels;table['membership_strength']=strength
                table['threshold_source']=sources;table['is_anomaly_candidate']=flags
                table.to_csv(OUT/f'{split}_{name}_{detector}_{policy}_scores.csv',index=False)
                metrics.append({'split':split,'model':name,'detector':detector,'policy':policy,
                    'windows':len(score),'mean_masked_mse':float(result['feature_errors'].mean()),
                    'flagged_windows':int(flags.sum()),'alert_fraction':float(flags.mean()),
                    'fallback_fraction':float((sources=='global_fallback').mean())})
metrics=pd.DataFrame(metrics);metrics.to_csv(OUT/'frozen_candidate_metrics.csv',index=False)
display(metrics.loc[metrics.split.eq('acceptance')])


,split,model,detector,policy,windows,mean_masked_mse,flagged_windows,alert_fraction,fallback_fraction
16,acceptance,frozen,R,global,2989,0.005344,0,0.000000,0.000000
17,acceptance,frozen,R,conditional,2989,0.005344,2,0.000669,0.513215
18,acceptance,frozen,E,global,2989,0.005344,14,0.004684,0.000000
19,acceptance,frozen,E,conditional,2989,0.005344,15,0.005018,0.513215
20,acceptance,frozen,D,global,2989,0.005344,0,0.000000,0.000000
21,acceptance,frozen,D,conditional,2989,0.005344,9,0.003011,0.513215
22,acceptance,frozen,RD,global,2989,0.005344,0,0.000000,0.000000
23,acceptance,frozen,RD,conditional,2989,0.005344,2,0.000669,0.513215
24,acceptance,candidate,R,global,2989,0.005195,0,0.000000,0.000000
25,acceptance,candidate,R,conditional,2989,0.005195,2,0.000669,0.512211


## 4. Check historical retention and fixed partition compatibility

Encoder updates can move embeddings away from the frozen scaler/PCA/clusters. We conservatively reject assignment changes above 5% on either evaluation group. Historical reconstruction is checked overall and within each sufficiently supported original partition. These additional compatibility limits were saved before opening acceptance scores.

Passing a 5% assignment-change check still does not prove unchanged geometry; normalized distance and fallback changes are reported separately. Full re-estimation of geometry would need a separately designed calibration experiment.

In [6]:
gates=[]
def gate(name,passed,observed,limit):gates.append({'gate':name,'passed':bool(passed),'observed':float(observed),'limit':float(limit)})
retention=[]
old_labels=SCORED[('old_replay','frozen')][1]
for label in ['all']+sorted(set(old_labels)-{-1}):
    select=np.ones(len(old_raw),dtype=bool) if label=='all' else old_labels==label
    drives=anchors.loc[select,'serial_number'].nunique()
    if label!='all' and (select.sum()<TRAINING['old_retention_minimum_partition_windows'] or drives<TRAINING['old_retention_minimum_partition_drives']):continue
    before=float(REP[('old_replay','frozen')]['feature_errors'][select].mean())
    after=float(REP[('old_replay','candidate')]['feature_errors'][select].mean())
    limit=before*POLICY['old_replay_mse_max_ratio']+POLICY['old_replay_mse_absolute_slack']
    gate('old_reconstruction_'+str(label),after<=limit,after,limit)
    retention.append({'partition':str(label),'windows':int(select.sum()),'drives':int(drives),'frozen_mse':before,'candidate_mse':after,'limit':limit,'passed':after<=limit})
retention=pd.DataFrame(retention);retention.to_csv(OUT/'old_replay_retention.csv',index=False)
geometry=[]
for split in ['old_replay','acceptance']:
    fs,fl,_,fc=SCORED[(split,'frozen')];cs,cl,_,cc=SCORED[(split,'candidate')]
    changed=float((fl!=cl).mean());gate(split+'_partition_assignment_change',changed<=TRAINING['maximum_partition_assignment_change_fraction'],changed,TRAINING['maximum_partition_assignment_change_fraction'])
    geometry.append({'split':split,'partition_assignment_change_fraction':changed,
        'frozen_median_D':float(np.median(fs['D'])),'candidate_median_D':float(np.median(cs['D'])),
        'frozen_unsupported_fraction':float((fc<0).mean()),'candidate_unsupported_fraction':float((cc<0).mean())})
for detector in ['R','E','D','RD']:
    for policy in ['global','conditional']:
        subset=metrics.loc[metrics.split.eq('acceptance')&metrics.detector.eq(detector)&metrics.policy.eq(policy)].set_index('model')
        increase=float(subset.loc['candidate','alert_fraction']-subset.loc['frozen','alert_fraction'])
        gate('acceptance_alert_increase_'+detector+'_'+policy,increase<=POLICY['acceptance_alert_fraction_max_increase'],increase,POLICY['acceptance_alert_fraction_max_increase'])
pd.DataFrame(geometry).to_csv(OUT/'partition_geometry_audit.csv',index=False)
display(retention)


,partition,windows,drives,frozen_mse,candidate_mse,limit,passed
0,all,4885,700,0.004702,0.004692,0.005174,True
1,0,101,20,0.004961,0.004815,0.005458,True
2,1,4224,645,0.004081,0.004074,0.004490,True


## 5. Retain synthetic sensitivity on the same development controls

Use the seven original Notebook 06 perturbations on the already inspected May development windows. A synthetic example is eligible only if its raw values actually change and the **frozen original** detector did not already flag it. Both models are evaluated on this same eligible set, so changed baseline alerts cannot improve the denominator artificially. Compare newly introduced alerts relative to each model's own unchanged-window flag.

A drop greater than five percentage points fails the declared tolerance. If the frozen model already misses a perturbation, preserving that zero sensitivity is not evidence that the candidate detects it; the limitation remains in the report.

In [7]:
with np.load(remember(original_root/'development/windows_raw.npz')) as a:dev_raw=a['X'].astype(np.float32)
base={name:cached_inference(name,'development_base',dev_raw) for name in STATES}
base_scored={name:frozen_scores(result) for name,result in base.items()}
base_flags={}
for name,(scores,_,_,context) in base_scored.items():
    for detector,values in scores.items():
        for policy in ['global','conditional']:base_flags[(name,detector,policy)]=threshold_flags(values,context,detector,policy)[0]
experiments=['temperature_plus_2_last_7_days','temperature_plus_5_last_7_days','temperature_spike_plus_10',
    'temperature_frozen_last_7_days','temperature_alignment_break','sector_5_plus_8_last_7_days','sector_197_198_plus_8_last_7_days']
def perturb(name):
    x=dev_raw.copy()
    if name=='temperature_plus_2_last_7_days':x[:,-7:,TEMPERATURE]+=2
    elif name=='temperature_plus_5_last_7_days':x[:,-7:,TEMPERATURE]+=5
    elif name=='temperature_spike_plus_10':x[:,-1,TEMPERATURE]+=10
    elif name=='temperature_frozen_last_7_days':x[:,-7:,TEMPERATURE]=x[:,-8,TEMPERATURE][:,None]
    elif name=='temperature_alignment_break':
        rng=np.random.default_rng(45)
        for i in range(len(x)):x[i,-7:,TEMPERATURE]=rng.permutation(x[i,-7:,TEMPERATURE])
    elif name=='sector_5_plus_8_last_7_days':x[:,-7:,FEATURES.index('smart_5_raw')]+=8
    elif name=='sector_197_198_plus_8_last_7_days':
        for feature in ['smart_197_raw','smart_198_raw']:x[:,-7:,FEATURES.index(feature)]+=8
    else:raise ValueError(name)
    return x
sensitivity=[]
for experiment in experiments:
    changed=perturb(experiment);actually_changed=(changed!=dev_raw).any(axis=(1,2))
    modified={name:frozen_scores(cached_inference(name,experiment,changed)) for name in STATES}
    for detector in ['R','E','D','RD']:
        for policy in ['global','conditional']:
            eligible=actually_changed & ~base_flags[('frozen',detector,policy)]
            rates={}
            for name,(scores,_,_,context) in modified.items():
                flags=threshold_flags(scores[detector],context,detector,policy)[0]
                new=flags & ~base_flags[(name,detector,policy)] & eligible
                rates[name]=float(new.sum()/eligible.sum()) if eligible.any() else None
            drop=rates['frozen']-rates['candidate'] if eligible.any() else None
            sensitivity.append({'experiment':experiment,'detector':detector,'policy':policy,'eligible_windows':int(eligible.sum()),
                'frozen_new_alert_rate':rates['frozen'],'candidate_new_alert_rate':rates['candidate'],
                'absolute_drop':drop,'frozen_sensitivity_absent':rates['frozen']==0,
                'passed':drop<=POLICY['synthetic_new_alert_rate_max_absolute_drop'] if drop is not None else False})
            # Empty eligibility is inconclusive and blocks a full pass rather than inventing a rate.
            gate('sensitivity_'+experiment+'_'+detector+'_'+policy,drop is not None and drop<=POLICY['synthetic_new_alert_rate_max_absolute_drop'],drop if drop is not None else 1.,POLICY['synthetic_new_alert_rate_max_absolute_drop'])
sensitivity=pd.DataFrame(sensitivity);sensitivity.to_csv(OUT/'synthetic_sensitivity_retention.csv',index=False)
display(sensitivity[['experiment','detector','policy','frozen_new_alert_rate','candidate_new_alert_rate','passed']])


,experiment,detector,policy,frozen_new_alert_rate,candidate_new_alert_rate,passed
0,temperature_plus_2_last_7_days,R,global,0.000000,0.000000,True
1,temperature_plus_2_last_7_days,R,conditional,0.000000,0.000000,True
2,temperature_plus_2_last_7_days,E,global,0.002380,0.001947,True
3,temperature_plus_2_last_7_days,E,conditional,0.001516,0.001516,True
4,temperature_plus_2_last_7_days,D,global,0.000000,0.000000,True
5,temperature_plus_2_last_7_days,D,conditional,0.000873,0.000436,True
6,temperature_plus_2_last_7_days,RD,global,0.000000,0.000000,True
7,temperature_plus_2_last_7_days,RD,conditional,0.000431,0.000431,True
8,temperature_plus_5_last_7_days,R,global,0.000000,0.000000,True
9,temperature_plus_5_last_7_days,R,conditional,0.000215,0.000215,True


## 6. Exercise rollback, record the decision and preserve the frozen model

The candidate checkpoint remains a separate research artifact. To exercise restoration even if all numerical gates pass, load the candidate, then restore the frozen state from disk and reproduce the reference outputs. A rollback test checks restoration; it does not establish production recovery time.

Any failed check rejects the candidate under this fixed protocol. A numerical pass permits further offline review only. A small pilot with one HDD model and one seed is insufficient for automatic deployment.

In [9]:
# Exercise an actual state transition: candidate -> frozen checkpoint loaded from disk.
model.load_state_dict(CANDIDATE_STATE)
restored=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
model.load_state_dict(restored['state_dict']);model.eval()
rollback_check=infer_raw(reference_raw)
for key in rollback_check:np.testing.assert_allclose(rollback_check[key],expected[key],rtol=1e-3,atol=1e-4)
assert sha256(checkpoint_path)==FROZEN_HASH and sha256(candidate_path)==CANDIDATE_HASH
gate('rollback_reference_reproduction',True,1,1)
checks=pd.DataFrame(gates);checks.to_csv(OUT/'acceptance_gates.csv',index=False)
passed=bool(checks.passed.all());decision='passed_pilot_checks_offline_review_only' if passed else 'rejected_by_fixed_protocol'
accept_before=float(REP[('acceptance','frozen')]['feature_errors'].mean())
accept_after=float(REP[('acceptance','candidate')]['feature_errors'].mean())
benefit={'frozen_acceptance_mse':accept_before,'candidate_acceptance_mse':accept_after,
    'acceptance_reconstruction_improved':bool(accept_after<accept_before),
    'interpretation':'Lower unlabelled reconstruction error does not establish better anomaly or failure detection.'}
for relative,digest in INPUTS.items():assert sha256(COMPONENT_ROOT/relative)==digest,f'Input changed during trial: {relative}'
result_record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,'run_id':RUN_ID,
    'experiment_id':AUDIT_ID,'decision':decision,'all_pilot_checks_passed':passed,
    'failed_gates':checks.loc[~checks.passed,'gate'].tolist(),'training_settings':TRAINING,
    'trial_identity':trial_identity,'frozen_checkpoint_sha256':FROZEN_HASH,'candidate_checkpoint_sha256':CANDIDATE_HASH,
    'input_sha256':INPUTS,'acceptance_comparisons':1,'acceptance_scores_inspected':True,
    'candidate_checkpoints_trained':1,'deployed_model_updates':0,'rollback_exercised':True,
    'active_in_memory_model':'frozen','sealed_test_opened':False,'failure_labels_used':False,
    'new_temporal_regime_validated':False,'drift_triggered_adaptation_validated':False,'reconstruction_comparison':benefit,
    'limitations':['one model/seed and one candidate','unlabelled acceptance scores are not failure performance',
        'historical retention uses training replay','zero baseline sector sensitivity remains a limitation',
        'frozen geometry compatibility is approximate','not an end-to-end chronological drift-triggered adaptation study']}
(OUT/'offline_adaptation_complete.json').write_text(json.dumps(result_record,indent=2,allow_nan=False))
display(checks.loc[~checks.passed])
print('NOTEBOOK 11 COMPLETE —',decision)
print('Failed checks:',int((~checks.passed).sum()))
print('Rollback exercised. Active in-memory model: frozen. Deployed updates: 0.')
print('September acceptance has now been inspected; do not retune against this reservation.')
print('Results:',OUT)


,gate,passed,observed,limit


NOTEBOOK 11 COMPLETE — passed_pilot_checks_offline_review_only
Failed checks: 0
Rollback exercised. Active in-memory model: frozen. Deployed updates: 0.
September acceptance has now been inspected; do not retune against this reservation.
Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\offline_replay_adaptation_v1
